In [ ]:
# !pip3 install bibtexparser

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
import bibtexparser
from bibtexparser.bparser import BibTexParser
from bibtexparser.customization import homogenize_latex_encoding
import csv
import re

def normalize_title(title):
    title = homogenize_latex_encoding({"title": title})["title"]
    """Normalize title for comparison: lowercase, remove spaces and punctuation."""
    return re.sub(r'\W+', '', title.lower().strip())

def load_bibtex_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as bibfile:
        parser = BibTexParser(common_strings=True)
        # parser.customization = homogenize_latex_encoding
        return bibtexparser.load(bibfile, parser=parser)

# Load both BibTeX files
bibtex = load_bibtex_file("google-scholar-selected.bib")

# Index entries by normalized title
bibtex_entries = {}

# Prefer detailed entries
for entry in bibtex.entries:
    title = entry.get("title", "")
    if not title or not entry.get("year"):
        continue
    norm_title = normalize_title(title)
    if norm_title in bibtex_entries:
        print(f"Duplicate entry found for title: {title}")
        # If already exists, prefer the one with more fields
        existing_entry = bibtex_entries[norm_title]
        print(f"Existing entry: {existing_entry.keys()}")
        print(f"New entry: {entry.keys()}")
        if len(entry) > len(existing_entry):
            bibtex_entries[norm_title] = entry
    bibtex_entries[norm_title] = entry

# Known journal/booktitle to abbreviation map
known_abbrs = {
    "Journal of Machine Learning Research": "JMLR",
    "IEEE Internet Computing": "IEEE Internet Comp",
    "IEEE Transactions on Pattern Analysis and Machine Intelligence": "TPAMI",
    "Transactions of the Association for Computational Linguistics": "TACL",
    "Computational Linguistics": "CL",
    "IEEE Computational Intelligence Magazine": "IEEE CIM",
    "IEEE Intelligent Systems": "IEEE IS",
    "IEEE Access": "IEEE Access",
    "ACM Transactions on Information Systems": "TOIS",
    "ACM Transactions on Intelligent Systems and Technology": "TIST",
    "IEEE Transactions on Knowledge and Data Engineering": "TKDE",
    "Pattern Recognition Letters": "PRL",
    "Neurocomputing": "Neurocomputing",
    "Expert Systems with Applications": "ESWA",
    "Annual Conference on Neural Information Processing Systems": "NeurIPS",
    "International Conference on Learning Representations": "ICLR",
    "International Conference on Machine Learning": "ICML",
    "International Joint Conference on Artificial Intelligence": "IJCAI",
    "AAAI Conference on Artificial Intelligence": "AAAI",
    "Conference on Empirical Methods in Natural Language Processing": "EMNLP",
    "Annual Meeting of the Association for Computational Linguistics": "ACL",
    "North American Chapter of the ACL": "NAACL",
    "Conference on Computational Linguistics": "COLING",
    "European Chapter of the ACL": "EACL",
    "Conference on Computer Vision and Pattern Recognition": "CVPR",
    "International Conference on Computer Vision": "ICCV",
    "European Conference on Computer Vision": "ECCV",
    "International Conference on Data Engineering": "ICDE",
    "Very Large Data Bases": "VLDB",
    "ACM SIGMOD International Conference on Management of Data": "SIGMOD",
    "Annual Conference on Web Search and Data Mining": "WSDM",
    "The Web Conference": "WWW",
    "International Conference on Natural Language Processing": "ICON",
    "Language Resources and Evaluation Conference": "LREC",
    "International Conference on Intelligent Text Processing and Computational Linguistics": "CICLing",
}

def infer_abbr(entry):
    if "abbr" not in entry:
        journal = entry.get("journal", "") or entry.get("booktitle", "") or entry.get("publisher", "")
        for known, abbr in known_abbrs.items():
            if known.lower() in journal.lower():
                entry["abbr"] = abbr
                # try:
                #     if int(entry.get("year", 0)) > 2020:
                #         entry["selected"] = "true"
                # except ValueError:
                #     pass  # Ignore non-integer years
                break

# Apply abbreviation inference to merged entries
for entry in bibtex_entries.values():
    infer_abbr(entry)
# Create output bib database
output_db = bibtexparser.bibdatabase.BibDatabase()
output_db.entries = list(bibtex_entries.values())

# Dump to papers.bib
writer = bibtexparser.bwriter.BibTexWriter()
writer.order_entries_by = None
with open("papers-selected.bib", "w", encoding="utf-8") as outfile:
    outfile.write(writer.write(output_db))

print(f"✔ Merged {len(output_db.entries)}/{len(bibtex.entries)} entries into papers-selected.bib")

# Collect all unique keys across all entries
# all_fields = set()
# for entry in output_db.entries:
#     all_fields.update(entry.keys())

# Optional: Move commonly used fields to the front
csv_fields = ["ID", "abbr", "selected", "title", "author", "year", "journal", "booktitle", "doi", "url", "pdf", "eprint"]
# csv_fields = csv_fields + sorted(all_fields - set(csv_fields))

# Output CSV to stdout or file
with open("papers-selected.csv", "w", encoding="utf-8", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_fields)
    writer.writeheader()
    for entry in output_db.entries:
        row = {field: entry.get(field, "") for field in csv_fields}
        writer.writerow(row)

print("✔ Save papers-selected.csv")


✔ Merged 63/63 entries into papers-selected.bib
✔ Save papers-selected.csv


In [1]:
import csv
import bibtexparser
from bibtexparser.bparser import BibTexParser
from bibtexparser.customization import homogenize_latex_encoding
import re

def normalize_title(title):
    title = homogenize_latex_encoding({"title": title})["title"]
    return re.sub(r'\W+', '', title.lower().strip())

def load_bibtex_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as bibfile:
        parser = BibTexParser(common_strings=True)
        return bibtexparser.load(bibfile, parser=parser)

# Load bib entries
bibtex = load_bibtex_file("papers-selected.bib")
entries = bibtex.entries

# Load CSV entries into a dict (indexed by ID or normalized title)
abbr_map = {}
with open("papers-selected.csv", newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        key = row.get("ID") or normalize_title(row.get("title", ""))
        if key and row.get("abbr"):
            abbr_map[key] = row["abbr"]

# Inject abbr field back into BibTeX entries
for entry in entries:
    key = entry.get("ID") or normalize_title(entry.get("title", ""))
    if key in abbr_map:
        entry["abbr"] = abbr_map[key]

# Save updated BibTeX
output_db = bibtexparser.bibdatabase.BibDatabase()
output_db.entries = entries

writer = bibtexparser.bwriter.BibTexWriter()
writer.order_entries_by = None
with open("papers-selected-updated.bib", "w", encoding="utf-8") as outfile:
    outfile.write(writer.write(output_db))

print("✔ Updated `abbr` fields injected and saved to papers-selected-updated.bib")


✔ Updated `abbr` fields injected and saved to papers-selected-updated.bib
